<a href="https://colab.research.google.com/github/QUANHONGLE/CS421-emotion-prediction/blob/main/notebooks_p2/Q1_Corpus_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers
!pip install rouge-score
!pip install bert-score
!pip install sacrebleu
!pip install scikit-learn
!pip install pandas
!pip install numpy
!pip install torch

In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

# Load data
train_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_train.csv')
dev_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_dev.csv', on_bad_lines='skip')
test_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/main/P1_data/trac2_CONVT_test.csv', on_bad_lines='skip')

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)
print(train_df.columns.tolist())

In [ ]:
# Load SBERT model
print("Loading SBERT model...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
print("SBERT loaded!")

# Compute embeddings for training data
print("Computing training embeddings...")
train_embeddings = sbert_model.encode(train_df['text'].astype(str).tolist(), show_progress_bar=True)
print("Embeddings done! Shape:", train_embeddings.shape)

# Normalize Emotion and Empathy scores to [0, 1]
emotion_min, emotion_max = train_df['Emotion'].min(), train_df['Emotion'].max()
empathy_min, empathy_max = train_df['Empathy'].min(), train_df['Empathy'].max()

train_df['Emotion_norm'] = (train_df['Emotion'] - emotion_min) / (emotion_max - emotion_min)
train_df['Empathy_norm'] = (train_df['Empathy'] - empathy_min) / (empathy_max - empathy_min)

# One-hot encode polarity (0, 1, 2, 3)
polarity_dummies = pd.get_dummies(train_df['EmotionalPolarity'], prefix='polarity')
train_df = pd.concat([train_df, polarity_dummies], axis=1)

print("Normalization done!")
print("Emotion range:", train_df['Emotion_norm'].min(), "-", train_df['Emotion_norm'].max())
print("Empathy range:", train_df['Empathy_norm'].min(), "-", train_df['Empathy_norm'].max())
print("Polarity columns:", polarity_dummies.columns.tolist())

In [ ]:
def calculate_similarity(query_embedding, query_emotion, query_empathy, query_polarity,
                          train_embeddings, train_emotion, train_empathy, train_polarity,
                          w1=0.6, w2=0.15, w3=0.15, w4=0.1):
    # Text similarity - cosine similarity between embeddings
    s_text = cosine_similarity([query_embedding], train_embeddings)[0]

    # Emotion similarity
    s_emotion = 1 - np.abs(train_emotion - query_emotion)

    # Empathy similarity
    s_empathy = 1 - np.abs(train_empathy - query_empathy)

    # Polarity similarity
    s_polarity = (train_polarity == query_polarity).astype(float)

    # Combined score
    s_total = w1 * s_text + w2 * s_emotion + w3 * s_empathy + w4 * s_polarity

    return s_total

# Test it works
sample_embedding = train_embeddings[0]
sample_emotion = train_df['Emotion_norm'].values[0]
sample_empathy = train_df['Empathy_norm'].values[0]
sample_polarity = train_df['EmotionalPolarity'].values[0]

scores = calculate_similarity(
    sample_embedding, sample_emotion, sample_empathy, sample_polarity,
    train_embeddings,
    train_df['Emotion_norm'].values,
    train_df['Empathy_norm'].values,
    train_df['EmotionalPolarity'].values
)

print("Similarity scores shape:", scores.shape)
print("Max score:", scores.max())
print("Min score:", scores.min())
print("Best match index:", scores.argmax())
print("Best match text:", train_df['text'].values[scores.argmax()])

In [ ]:
def get_query_embedding(conversation_history, sbert_model):
    # Concatenate all utterances in the conversation history
    query_text = ' '.join(conversation_history)
    return sbert_model.encode(query_text)

def generate_next_utterance(conversation_history, query_emotion, query_empathy, query_polarity,
                             train_embeddings, train_df, sbert_model, used_indices=set()):
    # Get embedding for conversation history
    query_embedding = get_query_embedding(conversation_history, sbert_model)

    # Normalize query emotion and empathy using training min/max
    query_emotion_norm = (query_emotion - emotion_min) / (emotion_max - emotion_min)
    query_empathy_norm = (query_empathy - empathy_min) / (empathy_max - empathy_min)

    # Calculate similarity scores
    scores = calculate_similarity(
        query_embedding, query_emotion_norm, query_empathy_norm, query_polarity,
        train_embeddings,
        train_df['Emotion_norm'].values,
        train_df['Empathy_norm'].values,
        train_df['EmotionalPolarity'].values
    )

    # Mask already used sentences to avoid repetition
    scores[list(used_indices)] = -1

    # Get best match
    best_idx = scores.argmax()
    used_indices.add(best_idx)

    return train_df['text'].values[best_idx], best_idx

def generate_conversation(conv_history, emotion_targets, empathy_targets, polarity_targets,
                           train_embeddings, train_df, sbert_model, n_turns=5):
    generated = []
    used_indices = set()

    for i in range(n_turns):
        utterance, idx = generate_next_utterance(
            conv_history,
            emotion_targets[i],
            empathy_targets[i],
            polarity_targets[i],
            train_embeddings,
            train_df,
            sbert_model,
            used_indices
        )
        generated.append(utterance)
        conv_history.append(utterance)  # add generated utterance to history

    return generated

print("Generation functions defined!")

In [ ]:
def get_target_values(df, conv_id, turn_start=6, n_turns=5):
    conv = df[df['conversation_id'] == conv_id].sort_values('turn_id')
    targets = conv[conv['turn_id'] >= turn_start].head(n_turns)

    if len(targets) == 0:
        # if no turns available use average of conversation
        emotion_targets = [conv['Emotion'].mean()] * n_turns
        empathy_targets = [conv['Empathy'].mean()] * n_turns
        polarity_targets = [conv['EmotionalPolarity'].mode()[0]] * n_turns
    else:
        emotion_targets = targets['Emotion'].tolist()
        empathy_targets = targets['Empathy'].tolist()
        polarity_targets = targets['EmotionalPolarity'].tolist()
        # pad if less than n_turns
        while len(emotion_targets) < n_turns:
            emotion_targets.append(emotion_targets[-1])
            empathy_targets.append(empathy_targets[-1])
            polarity_targets.append(polarity_targets[-1])

    return emotion_targets, empathy_targets, polarity_targets

print("Generating for dev set...")
dev_results = []

for conv_id in dev_df['conversation_id'].unique():
    conv = dev_df[dev_df['conversation_id'] == conv_id].sort_values('turn_id')

    # Get first 5 turns as history
    history = conv[conv['turn_id'] <= 5]['text'].astype(str).tolist()
    if len(history) == 0:
        continue

    # Get target emotion/empathy/polarity for turns 6-10
    emotion_targets, empathy_targets, polarity_targets = get_target_values(dev_df, conv_id)

    # Generate next 5 turns
    generated = generate_conversation(
        history.copy(),
        emotion_targets,
        empathy_targets,
        polarity_targets,
        train_embeddings,
        train_df,
        sbert_model
    )

    for turn_num, utterance in enumerate(generated, start=6):
        dev_results.append({
            'conversation_id': conv_id,
            'turn_number': turn_num,
            'generated_response': utterance
        })

dev_results_df = pd.DataFrame(dev_results)
print("Dev generation done!")
print(f"Total generated turns: {len(dev_results_df)}")
print(dev_results_df.head(10))

In [ ]:
!pip install evaluate
import evaluate

# Load metrics
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')
bertscore = evaluate.load('bertscore')

# Get gold responses for dev set (turns 6-10)
gold_responses = []
pred_responses = []

for conv_id in dev_df['conversation_id'].unique():
    conv = dev_df[dev_df['conversation_id'] == conv_id].sort_values('turn_id')
    gold_turns = conv[conv['turn_id'] >= 6].head(5)['text'].astype(str).tolist()
    pred_turns = dev_results_df[dev_results_df['conversation_id'] == conv_id]['generated_response'].tolist()

    min_len = min(len(gold_turns), len(pred_turns))
    gold_responses.extend(gold_turns[:min_len])
    pred_responses.extend(pred_turns[:min_len])

# ROUGE
rouge_scores = rouge.compute(predictions=pred_responses, references=gold_responses)
print("=== ROUGE ===")
print(f"ROUGE-1: {rouge_scores['rouge1']:.4f}")
print(f"ROUGE-2: {rouge_scores['rouge2']:.4f}")
print(f"ROUGE-L: {rouge_scores['rougeL']:.4f}")

# BLEU
bleu_score = bleu.compute(predictions=pred_responses, references=[[g] for g in gold_responses])
print("\n=== BLEU ===")
print(f"BLEU: {bleu_score['bleu']:.4f}")

# BertScore
print("\nComputing BertScore (this may take a minute)...")
bert_scores = bertscore.compute(predictions=pred_responses, references=gold_responses, lang='en')
print("=== BertScore ===")
print(f"Precision: {sum(bert_scores['precision'])/len(bert_scores['precision']):.4f}")
print(f"Recall: {sum(bert_scores['recall'])/len(bert_scores['recall']):.4f}")
print(f"F1: {sum(bert_scores['f1'])/len(bert_scores['f1']):.4f}")

In [ ]:
test_p2_df = pd.read_csv('https://raw.githubusercontent.com/QUANHONGLE/CS421-emotion-prediction/refs/heads/main/P2%20Test%20Data/project_part2_test.csv', on_bad_lines='skip')
print(test_p2_df.shape)
print(test_p2_df.columns.tolist())
print(test_p2_df.head(3))

In [ ]:
print("Generating for test set...")
test_results = []

for conv_id in test_p2_df['conversation_id'].unique():
    conv = test_p2_df[test_p2_df['conversation_id'] == conv_id].sort_values('turn_id')

    # Get first 5 turns as history
    history = conv[conv['turn_id'] <= 5]['text'].astype(str).tolist()
    if len(history) == 0:
        continue

    # Get target emotion/empathy/polarity for turns 6-10
    emotion_targets, empathy_targets, polarity_targets = get_target_values(test_p2_df, conv_id)

    # Generate next 5 turns
    generated = generate_conversation(
        history.copy(),
        emotion_targets,
        empathy_targets,
        polarity_targets,
        train_embeddings,
        train_df,
        sbert_model
    )

    for turn_num, utterance in enumerate(generated, start=6):
        test_results.append({
            'conversation_id': conv_id,
            'turn_number': turn_num,
            'generated_response': utterance
        })

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv('generations_corpus.csv', index=False)
print("Saved generations_corpus.csv!")
print(f"Total generated turns: {len(test_results_df)}")
print(test_results_df.head(10))

In [ ]:
print("Generating 5 conversations with 10 utterances each...")
long_results = []

# Pick 5 conversation ids from dev set
sample_conv_ids = dev_df['conversation_id'].unique()[:5]

for conv_id in sample_conv_ids:
    conv = dev_df[dev_df['conversation_id'] == conv_id].sort_values('turn_id')

    # Get first 5 turns as history
    history = conv[conv['turn_id'] <= 5]['text'].astype(str).tolist()
    if len(history) == 0:
        continue

    # Get target values for 10 turns
    emotion_targets, empathy_targets, polarity_targets = get_target_values(dev_df, conv_id, turn_start=6, n_turns=10)

    # Generate next 10 turns
    generated = generate_conversation(
        history.copy(),
        emotion_targets,
        empathy_targets,
        polarity_targets,
        train_embeddings,
        train_df,
        sbert_model,
        n_turns=10
    )

    for turn_num, utterance in enumerate(generated, start=6):
        long_results.append({
            'conversation_id': conv_id,
            'turn_number': turn_num,
            'generated_response': utterance
        })

long_results_df = pd.DataFrame(long_results)
long_results_df.to_csv('generations_corpus_10turns.csv', index=False)
print("Saved generations_corpus_10turns.csv!")
print(long_results_df.to_string())

In [ ]:
def evaluate_weights(w1, w2, w3, w4, dev_df, train_df, train_embeddings, sbert_model):
    results = []
    for conv_id in list(dev_df['conversation_id'].unique())[:20]:  # test on 20 convos for speed
        conv = dev_df[dev_df['conversation_id'] == conv_id].sort_values('turn_id')
        history = conv[conv['turn_id'] <= 5]['text'].astype(str).tolist()
        if len(history) == 0:
            continue
        emotion_targets, empathy_targets, polarity_targets = get_target_values(dev_df, conv_id)
        generated = generate_conversation(
            history.copy(), emotion_targets, empathy_targets, polarity_targets,
            train_embeddings, train_df, sbert_model, n_turns=5
        )
        for turn_num, utterance in enumerate(generated, start=6):
            results.append({'conversation_id': conv_id, 'turn_number': turn_num, 'generated_response': utterance})
    return pd.DataFrame(results)

weight_configs = [
    (0.6, 0.15, 0.15, 0.1),   # our original
    (0.25, 0.25, 0.25, 0.25), # equal weights
    (0.7, 0.1, 0.1, 0.1),     # text heavy
    (0.4, 0.2, 0.2, 0.2),     # balanced with less text
]

for w1, w2, w3, w4 in weight_configs:
    results_df = evaluate_weights(w1, w2, w3, w4, dev_df, train_df, train_embeddings, sbert_model)
    gold = []
    pred = []
    for conv_id in results_df['conversation_id'].unique():
        conv = dev_df[dev_df['conversation_id'] == conv_id].sort_values('turn_id')
        gold_turns = conv[conv['turn_id'] >= 6].head(5)['text'].astype(str).tolist()
        pred_turns = results_df[results_df['conversation_id'] == conv_id]['generated_response'].tolist()
        min_len = min(len(gold_turns), len(pred_turns))
        gold.extend(gold_turns[:min_len])
        pred.extend(pred_turns[:min_len])

    r = rouge.compute(predictions=pred, references=gold)
    b = bertscore.compute(predictions=pred, references=gold, lang='en')
    f1 = sum(b['f1']) / len(b['f1'])
    print(f"w1={w1} w2={w2} w3={w3} w4={w4} | ROUGE-1: {r['rouge1']:.4f} | BertScore F1: {f1:.4f}")